# 03 — Case Studies and Mechanisms

Four pieces of supplementary evidence. None is offered as a causal estimate of
the Convention's effect; each has a different inferential role, and each is kept
out of the main analysis for that reason.

* **Part A — Turkey's withdrawal.** One treated unit. Descriptive.
* **Part B — Synthetic control.** A feasibility result: the method cannot be
  applied to the countries of primary interest, for a substantive reason.
* **Part C — Recorded sexual violence.** A reporting mechanism, not incidence.
* **Part D — Institutional outcomes.** Descriptive legislative event histories.

**Input** `data/gbv_panel_analysis.csv`.
**Output** `outputs/tables/`, two figures in `outputs/figures/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "src"))

from analysis_helpers import *          # noqa: F401,F403
import estimators as E
import scm as SCM

DATA_PATH = resolve_data_path(PROJECT_DIR)
OUT = make_output_dirs(PROJECT_DIR)
_d = load_and_prepare_data(DATA_PATH, OUT)
df, SAMPLES = _d["df"], _d["SAMPLES"]
country_info, inventory = _d["country_info"], _d["inventory"]
ALL_CTRL = _d["ALL_CTRL"]

## Part A — Turkey's withdrawal from the Convention

Turkey ratified in 2012, and its withdrawal took legal effect on 1 July 2021.
It is the only treatment reversal in the panel, which makes it substantively
interesting and statistically intractable: there is exactly one treated unit.

Three things are reported together, because the first is misleading without the
other two:

1. the conventional clustered estimate;
2. that it rests on a single treated cluster, so its p-value is not valid;
3. that Turkey was already on a divergent trend before the withdrawal, and that
   its series contains an unexplained break in 2013–2014.

In [ ]:
tp = build_turkey_panel(df, "fhr")
print(f"panel: {len(tp)} rows, {tp.country.nunique()} countries, "
      f"treated clusters = {tp.loc[tp.is_turkey==1,'country'].nunique()}")

r_cl = E.feols(tp, "fhr", ["did_withdrawal"], ["country","year"],
               cluster="country", name="turkey clustered")
r_hc = E.feols(tp, "fhr", ["did_withdrawal"], ["country","year"],
               cluster=None, name="turkey unclustered")
print(f"\nclustered on country (1 treated cluster): b={r_cl['did_withdrawal']['coef']:+.4f} "
      f"SE={r_cl['did_withdrawal']['se']:.4f} p={r_cl['did_withdrawal']['p']:.2e}")
print(f"no clustering (heteroskedasticity-robust): b={r_hc['did_withdrawal']['coef']:+.4f} "
      f"SE={r_hc['did_withdrawal']['se']:.4f} p={r_hc['did_withdrawal']['p']:.4f}")
print("\nThe clustered p-value below is an artefact of clustering with a")
print("single treated cluster. It is not evidence of anything.")

In [ ]:
# Pre-withdrawal differential trend, against the same comparison group.
pre = tp[tp.year < 2021].copy()
pre["turkey_trend"] = pre.is_turkey * pre.year
r_pre = E.feols(pre, "fhr", ["turkey_trend"], ["country","year"],
                cluster="country", name="turkey pretrend")
print(f"Turkey differential trend 2012-2020: b={r_pre['turkey_trend']['coef']:+.4f}/yr "
      f"SE={r_pre['turkey_trend']['se']:.4f} p={r_pre['turkey_trend']['p']:.4f}")
print("\nTurkey was already diverging from the comparison group before withdrawal.")
print("A DiD that assumes parallel trends is not identified here.")

In [ ]:
# Placebo reassignment: relabel each comparison country as the withdrawer.
actual_b = r_cl["did_withdrawal"]["coef"]
betas = []
for c in sorted(tp.loc[tp.is_turkey==0,"country"].unique()):
    pp = tp.copy()
    pp["did_placebo"] = ((pp.country == c) & (pp.post_withdrawal == 1)).astype(float)
    if pp.did_placebo.nunique() < 2:
        continue
    try:
        rp = E.feols(pp, "fhr", ["did_placebo"], ["country","year"],
                     cluster=None, name="placebo")
        betas.append(rp["did_placebo"]["coef"])
    except RuntimeError:
        continue
betas = np.array(betas)
rank_p = float((np.abs(betas) >= abs(actual_b)).sum() + 1) / (len(betas) + 1)
print(f"Turkey coefficient {actual_b:+.4f}; placebo distribution n={len(betas)}, "
      f"mean={betas.mean():+.4f}, sd={betas.std(ddof=1):.4f}")
print(f"Rank p-value against placebo reassignment: {rank_p:.4f}")
print("\nTurkey is not unusual against countries that did not withdraw.")
pd.DataFrame({"placebo_beta": betas}).to_csv(
    OUT["tables"] / "turkey_placebo_reassignment.csv", index=False)

In [ ]:
# What is actually driving the coefficient?
tur = df[(df.country=="Turkey") & df.fhr.notna()][["year","fhr"]]
print("Turkey female homicide by year:")
print(tur.to_string(index=False))
print("\n2021 -> 2023 change:",
      f"{tur.loc[tur.year==2023,'fhr'].iloc[0] - tur.loc[tur.year==2021,'fhr'].iloc[0]:+.3f}",
      "of which the 2022->2023 step alone is",
      f"{tur.loc[tur.year==2023,'fhr'].iloc[0] - tur.loc[tur.year==2022,'fhr'].iloc[0]:+.3f}")
print("Missing years 2013 and 2014 sit between ratification (2012) and entry")
print("into force (2014); the level falls 36% across that gap.")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.5))
ctrl = tp[tp.is_turkey==0].groupby("year").fhr.mean()
tsub = tur[tur.year >= 2012]
ax.plot(tsub.year, tsub.fhr, "o-", color=C_POLICY, linewidth=2.5, markersize=7,
        label="Turkey", zorder=5)
ax.plot(ctrl.index, ctrl.values, "s--", color=C_TREATED, linewidth=2, markersize=5,
        alpha=.85, label="Still-active ratifiers (mean)", zorder=4)
ax.axvspan(2013, 2014, color="grey", alpha=.20, zorder=0)
ax.text(2013.5, ax.get_ylim()[1]*0.97, "no data\n2013–14", ha="center", va="top",
        fontsize=9, color="#444")
ax.axvline(2021, color=C_POLICY, linestyle="--", linewidth=2, alpha=.8)
ax.text(2021.1, ax.get_ylim()[1]*0.90, "withdrawal", color=C_POLICY, fontsize=10,
        fontweight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("Female homicide rate per 100,000")
ax.set_title("Turkey vs still-active ratifiers — descriptive, not causal")
ax.legend(fontsize=10); ax.grid(alpha=.25)
fig_note(f"Clustered p-value invalid (1 treated cluster). Unclustered p="
         f"{r_hc['did_withdrawal']['p']:.3f}; placebo-reassignment rank p={rank_p:.3f}. "
         f"Grey band = missing data.")
save_fig(OUT["figures"], "turkey_withdrawal_descriptive.png")

In [ ]:
# The legal outcomes cannot support a withdrawal DiD.
tl = df[(df.country=="Turkey") & (df.year>=2016)][["year","wbl_dv_legislation","wbl_femicide_law"]]
print(tl.to_string(index=False))
print("\nwbl_dv_legislation is constant at 1 -> a DiD coefficient on it reflects")
print("only the control countries. wbl_femicide_law switches 0->1 in 2023, i.e.")
print("Turkey adopted a femicide law AFTER withdrawing. Neither is a withdrawal")
print("effect. Both were dropped from the pipeline.")

## Part B — Synthetic control

A synthetic control builds a counterfactual as a weighted average of untreated
countries. Its credibility rests entirely on reproducing the treated country's
pre-treatment path.

The obstacle here is structural rather than computational. Every country that
never ratifies within the observation window has a higher female homicide rate
than the Western European ratifiers of primary interest, and a convex
combination cannot produce a value below all of its inputs. This follows from
studying a largely Western European treaty when the only available controls are
post-socialist non-parties, and it is reported as a finding about the research
design rather than hidden as a failed attempt.

In [ ]:
ALL_YEARS = sorted([y for y in df.year.unique() if y >= 2001])
feas = SCM.feasibility_report(df, [("Spain", 2014), ("Italy", 2013)], "fhr", ALL_YEARS)
print(feas.to_string(index=False))
print("\ndonors_below_treated = 0 for both: every eligible donor has a HIGHER")
print("pre-treatment homicide level than Spain and Italy, so no convex")
print("combination of donors can reach the treated country's level.")

In [ ]:
# Fit Spain and report the diagnostics rather than only reporting a refusal.
# The donor weights and the pre-period fit are themselves the evidence.
scm_rows = []
res, wts, dg = SCM.run_scm(df, "Spain", 2014, "fhr", ALL_YEARS)
scm_rows.append({k: v for k, v in dg.items() if not isinstance(v, dict)})
wts.to_csv(OUT["tables"] / "scm_spain_donor_weights.csv", index=False)
res.to_csv(OUT["tables"] / "scm_spain_series.csv", index=False)
print(wts.to_string(index=False))
pd.DataFrame(scm_rows).to_csv(OUT["tables"] / "scm_spain_diagnostics.csv", index=False)

In [ ]:
# Leave-one-donor-out. If dropping a single donor changes the sign of the
# estimated gap, the gap is a property of that donor rather than of treatment.
loo = SCM.leave_one_donor_out(df, "Spain", 2014, "fhr", ALL_YEARS, wts)
print(loo.to_string(index=False))
print(f"\nbaseline average post-treatment gap = {dg['avg_post_gap']:+.4f}")
loo.to_csv(OUT["tables"] / "scm_spain_leave_one_donor_out.csv", index=False)

In [ ]:
# Placebo in space: refit treating each donor as if it had been treated in 2014,
# then rank Spain's post/pre RMSPE ratio among them.
pl, p_rank = SCM.placebo_in_space(df, "Spain", 2014, "fhr", ALL_YEARS,
                                  actual_ratio=dg["rmspe_ratio_post_pre"])
print(f"Spain post/pre RMSPE ratio = {dg['rmspe_ratio_post_pre']:.4f}")
print(f"placebo-in-space rank p = {p_rank:.4f}  (n={len(pl)} placebo units)")
print()
print("This p-value is not interpretable when the pre-period fit is poor: a large")
print("post/pre ratio can reflect a bad pre-period fit rather than an effect.")
pl.drop(columns=["gaps"]).to_csv(
    OUT["tables"] / "scm_spain_placebo_in_space.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
rr = res.dropna(subset=["synthetic"])
ax.plot(res.year, res.actual, "o-", color=C_POLICY, linewidth=2.4, markersize=5,
        label="Spain (actual)")
ax.plot(rr.year, rr.synthetic, "s--", color="#888", linewidth=2, markersize=4,
        label="Synthetic Spain")
ax.axvline(2014, color=C_TREATED, linestyle="--", linewidth=1.8, alpha=.8)
ax.text(2014.15, ax.get_ylim()[1] * 0.94, "Ratification", color=C_TREATED, fontsize=10)
ax.set_xlabel("Year")
ax.set_ylabel("Female homicide rate per 100,000 population")
ax.set_title("Synthetic Spain does not reproduce Spain before treatment\n"
             f"pre-period R-squared = {dg['pre_period_r2']:.3f}", fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=.25)
fig_note("Donors restricted to countries that never ratify within the observation "
         "window. Synthetic series omitted where a weighted donor is missing.")
save_fig(OUT["figures"], "scm_spain_fit.png")

### The countries that lie inside the donor hull

Feasibility is a property of the treated country, not of the outcome. The
higher-rate ratifiers do sit inside the range spanned by the donors and can
therefore be fitted. They are reported whether or not the fit proves usable.

In [ ]:
rows = []
for c, ty in [("Serbia", 2013), ("Montenegro", 2013), ("Albania", 2013)]:
    try:
        res, w, dg = SCM.run_scm(df, c, ty, "fhr", ALL_YEARS, verbose=True)
        rows.append({"country": c, **{k: v for k, v in dg.items()
                                      if not isinstance(v, dict)}})
        w.to_csv(OUT["tables"] / f"scm_{c.lower()}_donor_weights.csv", index=False)
        res.to_csv(OUT["tables"] / f"scm_{c.lower()}_series.csv", index=False)
    except RuntimeError as e:
        print(f"  {c}: REFUSED - {e}")
        rows.append({"country": c, "fit_acceptable": False, "note": str(e)})
inhull = pd.DataFrame(rows)
inhull.to_csv(OUT["tables"] / "scm_in_hull_countries.csv", index=False)
print()
print("Best achievable fit in the project is Serbia (pre-period R2 = 0.533,")
print("RMSPE 0.68x its own pre-period SD, 0% of donor weight on later ratifiers).")
print("That is far better than Spain (R2 = 0.05) but still short of a threshold")
print("that would support causal interpretation. Report it as a borderline case,")
print("not as a result.")


## Part C — Recorded sexual violence

Recorded sexual violence measures reports to the police, not incidence, and
Articles 18, 21 and 55 of the Convention require countries to make reporting
easier. The treatment therefore plausibly changes the measurement of the
outcome, and an increase is as consistent with improved reporting as with more
violence.

The level specification is dominated by a small number of very high-recording
countries, so the outcome is analysed in logs. The result speaks to reporting
behaviour, not to violence.

In [ ]:
s = SAMPLES["svr"].copy()
s["log_svr"] = np.log(s.svr.where(s.svr > 0))
s = s.dropna(subset=["log_svr"])
r_lvl = E.feols(SAMPLES["svr"], "svr", ["did_interaction"], ["country","year"],
                cluster="country", name="svr level")
r_log = E.feols(s, "log_svr", ["did_interaction"], ["country","year"],
                cluster="country", name="svr log")
rows = [E.coef_row(r_lvl, "did_interaction", "Sexual violence — level (per 100k)"),
        E.coef_row(r_log, "did_interaction", "Sexual violence — log")]
t = export_table(rows, OUT["tables"] / "mechanism_sexual_violence.csv", "SV")
print(t[["label","b","se","p","N"]].to_string(index=False))
print(f"\nlog specification implies a {100*(np.exp(r_log['did_interaction']['coef'])-1):+.1f}% "
      "change in the RECORDED rate.")
print("Top recorded rates (these dominate the level specification):")
print(SAMPLES["svr"].groupby("country").svr.mean().nlargest(5).round(1).to_string())

## Part D — Institutional outcomes

Whether a country has domestic violence legislation, and whether it has a
specific femicide offence, from the World Bank Women, Business and the Law
database. Both are absorbing binary indicators and neither supports a
difference-in-differences design: nearly all adoption predates the adopting
country's own ratification, and no control country ever adopts a femicide law,
so there is no counterfactual.

They are reported as descriptive event histories. The window sensitivity below
shows that a DiD estimate on such a variable is determined by how much
pre-Convention history the sample happens to include, which is why none is
reported as an effect.

In [ ]:
fs = SAMPLES["wbl_femicide_law"]
rows = []
for c in sorted(fs.country.unique()):
    g = fs[fs.country==c]
    if g.wbl_femicide_law.max() == 1:
        fy = int(g.loc[g.wbl_femicide_law==1, "year"].min())
        ry = g.convention_ratified_year.iloc[0]
        rows.append({"country": c, "femicide_law_year": fy,
                     "ratification_year": (int(ry) if pd.notna(ry) else None),
                     "years_relative_to_ratification": (fy-int(ry) if pd.notna(ry) else None)})
fem_hist = pd.DataFrame(rows).sort_values("femicide_law_year")
fem_hist.to_csv(OUT["tables"] / "descriptive_femicide_law_event_history.csv", index=False)
print(fem_hist.to_string(index=False))
print(f"\ncontrol countries ever adopting a femicide law: "
      f"{sorted(fs.loc[(fs.treated_ever==0)&(fs.wbl_femicide_law==1),'country'].unique())}")
print("\nSensitivity of the DiD estimate to how much pre-Convention history is included:")
for y0 in [1990, 2000, 2010]:
    sub = fs[fs.year >= y0]
    r = E.feols(sub, "wbl_femicide_law", ["did_interaction"], ["country","year"],
                cluster="country", name=f"fem {y0}")
    rr = r["did_interaction"]
    print(f"  {y0}-2023: b={rr['coef']:+.4f} SE={rr['se']:.4f} p={rr['p']:.4f}")
print("\nThe estimate is a function of how much pre-Convention history is included.")
print("It is not reported as a causal effect.")

In [ ]:
print("\nCase studies and mechanisms complete.")

## Consolidated synthetic-control diagnostics

`outputs/tables/scm_diagnostics.csv` gathers the feasibility test and every
fitted synthetic in one place, with the fit statistics that determine whether
each may be interpreted.

In [ ]:
scm_all = []
for name, path in [("Spain", OUT["tables"] / "scm_spain_diagnostics.csv"),
                   ("in-hull countries", OUT["tables"] / "scm_in_hull_countries.csv")]:
    t = pd.read_csv(path)
    if "country" not in t.columns:
        t.insert(0, "country", name)
    scm_all.append(t)
scm_diag = pd.concat(scm_all, ignore_index=True)
keep = [c for c in ["country", "n_eligible_donors", "pre_period_r2",
                    "rmspe_ratio_vs_pre_sd", "share_weight_on_later_ratifiers",
                    "avg_post_gap", "fit_acceptable"] if c in scm_diag.columns]
scm_diag = scm_diag[keep]
scm_diag.to_csv(OUT["tables"] / "scm_diagnostics.csv", index=False)
print(scm_diag.round(4).to_string(index=False))
print()
print("fit_acceptable is False for every country: no synthetic control in this")
print("project reproduces its treated unit well enough before treatment to")
print("support a causal reading of the post-treatment gap.")